# Solutions – 3. Bayesian Parameter Estimation

This notebook provides worked solutions to the five exercises from
[03_bayesian_fit.ipynb](../Stats/03_bayesian_fit.ipynb).

We reuse the same dataset and `emcee` setup from notebook 03.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm, expon, beta as beta_dist
from scipy.optimize import minimize
import emcee
import corner

rng = np.random.default_rng(seed=123)

In [ ]:
# ── True parameters ──────────────────────────────────────────────────────────
theta_true  = 0.3
mu_true     = 5.0
sigma_true  = 0.5
lambda_true = 0.5
n_samples   = 2000

# ── Dataset ───────────────────────────────────────────────────────────────────
n_signal     = rng.binomial(n_samples, theta_true)
n_background = n_samples - n_signal

x_signal     = rng.normal(mu_true, sigma_true, size=n_signal)
x_background = rng.exponential(1.0 / lambda_true, size=n_background)

x_data = np.concatenate([x_signal, x_background])
rng.shuffle(x_data)

print(f"Generated {len(x_data)} events  ({n_signal} signal + {n_background} background)")

In [ ]:
# ── Log-likelihood ────────────────────────────────────────────────────────────
def log_likelihood(params, data):
    theta, mu, sigma, lam = params
    ls = np.log(theta)       + norm.logpdf(data, mu, sigma)
    lb = np.log(1.0 - theta) + expon.logpdf(data, scale=1.0/lam)
    return np.sum(np.logaddexp(ls, lb))

# ── Default (uniform) log-prior ────────────────────────────────────────────────
def log_prior_uniform(params):
    theta, mu, sigma, lam = params
    if not (0.0 < theta < 1.0): return -np.inf
    if not (0.0 < mu    < 14.0): return -np.inf
    if not (0.01 < sigma < 5.0): return -np.inf
    if not (0.01 < lam   < 5.0): return -np.inf
    return 0.0

def log_posterior_uniform(params, data):
    lp = log_prior_uniform(params)
    if not np.isfinite(lp): return -np.inf
    return lp + log_likelihood(params, data)

# ── MAP estimate as starting point ────────────────────────────────────────────
def map_estimate(log_post, data, p0=None):
    if p0 is None:
        p0 = [0.2, data.mean(), 0.6, 0.6]
    res = minimize(lambda p: -log_post(p, data), x0=p0, method='Nelder-Mead',
                   options={'xatol':1e-6,'fatol':1e-6,'maxiter':50000})
    return res.x

map_unif = map_estimate(log_posterior_uniform, x_data)
print("MAP (uniform prior):", map_unif)

In [ ]:
# -- MCMC helper --------------------------------------------------------------
def run_mcmc(log_post, data, p0_map, nwalkers=32, nburn=300, nsteps=3000, seed=0):
    ndim = len(p0_map)
    rng_mc = np.random.default_rng(seed)
    p0_walkers = p0_map + 1e-3 * rng_mc.standard_normal((nwalkers, ndim))
    sampler = emcee.EnsembleSampler(nwalkers, ndim, log_post, args=(data,))
    state = sampler.run_mcmc(p0_walkers, nburn, progress=False)
    sampler.reset()
    sampler.run_mcmc(state, nsteps, progress=False)
    return sampler.get_chain(flat=True)

flat_unif = run_mcmc(log_posterior_uniform, x_data, map_unif)
print(f"MCMC (uniform prior): {flat_unif.shape[0]} samples")

---
## Exercise 1 – Prior sensitivity: Beta(2,5) prior on $\theta$

**Task:** Replace the uniform prior on $\theta$ with $\text{Beta}(2,5)$ and compare
the marginal posterior for $\theta$.  What happens as $n \to \infty$?

In [ ]:
from scipy.stats import beta as beta_rv

def log_prior_beta(params):
    theta, mu, sigma, lam = params
    if not (0.0 < theta < 1.0):  return -np.inf
    if not (0.0 < mu    < 14.0): return -np.inf
    if not (0.01 < sigma < 5.0): return -np.inf
    if not (0.01 < lam   < 5.0): return -np.inf
    # Beta(2,5) prior on theta
    return beta_rv.logpdf(theta, a=2, b=5)

def log_posterior_beta(params, data):
    lp = log_prior_beta(params)
    if not np.isfinite(lp): return -np.inf
    return lp + log_likelihood(params, data)

# MAP and MCMC with Beta prior
map_beta  = map_estimate(log_posterior_beta, x_data, p0=map_unif)
flat_beta = run_mcmc(log_posterior_beta, x_data, map_beta)

theta_unif = flat_unif[:, 0]
theta_beta = flat_beta[:, 0]

print(f"Uniform prior: median = {np.median(theta_unif):.4f}, "
      f"68% CI = [{np.percentile(theta_unif,16):.4f}, {np.percentile(theta_unif,84):.4f}]")
print(f"Beta(2,5) prior: median = {np.median(theta_beta):.4f}, "
      f"68% CI = [{np.percentile(theta_beta,16):.4f}, {np.percentile(theta_beta,84):.4f}]")

In [ ]:
# Compare posteriors for two sample sizes
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
t = np.linspace(0.0, 1.0, 400)
beta_prior_pdf = beta_rv.pdf(t, a=2, b=5)

for ax, n in zip(axes, [200, 2000]):
    rng_loc = np.random.default_rng(seed=1)
    n_sig = rng_loc.binomial(n, theta_true)
    xs = np.concatenate([
        rng_loc.normal(mu_true, sigma_true, n_sig),
        rng_loc.exponential(1.0/lambda_true, n - n_sig)
    ])
    rng_loc.shuffle(xs)

    map_u = map_estimate(log_posterior_uniform, xs)
    map_b = map_estimate(log_posterior_beta, xs, p0=map_u)
    fu = run_mcmc(log_posterior_uniform, xs, map_u, nsteps=1500)
    fb = run_mcmc(log_posterior_beta, xs, map_b, nsteps=1500)

    ax.hist(fu[:, 0], bins=50, density=True, histtype='step', color='steelblue', lw=2,
            label='Uniform prior')
    ax.hist(fb[:, 0], bins=50, density=True, histtype='step', color='darkorange', lw=2,
            label='Beta(2,5) prior')
    ax.plot(t, beta_prior_pdf, 'g--', lw=1.5, label='Beta(2,5) prior PDF')
    ax.axvline(theta_true, color='red', ls=':', lw=1.5, label=f'True θ = {theta_true}')
    ax.set_xlabel(r'$\theta$'); ax.set_ylabel('Density'); ax.set_title(f'n = {n}')
    ax.legend(fontsize=8)

plt.suptitle('Prior sensitivity: Uniform vs. Beta(2,5)', fontsize=13)
plt.tight_layout(); plt.show()

**Observations:**
- At $n = 200$ the Beta(2,5) prior noticeably pulls the posterior toward lower values of
  $\theta$ compared to the flat prior, because the likelihood is not yet overwhelmingly
  informative.
- At $n = 2000$ the two posteriors almost coincide: with many data points the likelihood
  dominates and the prior becomes irrelevant.
- This illustrates the **Bernstein–von Mises theorem**: as $n \to \infty$, the posterior
  converges to a Gaussian centred at the MLE, regardless of the prior (under regularity
  conditions).

---
## Exercise 2 – Informative prior on $\mu$

**Task:** Encode a previous measurement $\mu_{\rm prev} = 5.1 \pm 0.2$ as a Gaussian
prior on $\mu$ and study how the posterior changes.

In [ ]:
mu_prev   = 5.1
sigma_prev = 0.2

def log_prior_info_mu(params):
    theta, mu, sigma, lam = params
    if not (0.0 < theta < 1.0):  return -np.inf
    if not (0.0 < mu    < 14.0): return -np.inf
    if not (0.01 < sigma < 5.0): return -np.inf
    if not (0.01 < lam   < 5.0): return -np.inf
    # Gaussian prior on mu
    return norm.logpdf(mu, mu_prev, sigma_prev)

def log_posterior_info_mu(params, data):
    lp = log_prior_info_mu(params)
    if not np.isfinite(lp): return -np.inf
    return lp + log_likelihood(params, data)

map_info = map_estimate(log_posterior_info_mu, x_data, p0=map_unif)
flat_info = run_mcmc(log_posterior_info_mu, x_data, map_info)

param_names = [r'$\theta$', r'$\mu$', r'$\sigma$', r'$\lambda$']
true_values = [theta_true, mu_true, sigma_true, lambda_true]

for i, pn in enumerate(param_names):
    u_med = np.median(flat_unif[:, i])
    i_med = np.median(flat_info[:, i])
    u_std = flat_unif[:, i].std()
    i_std = flat_info[:, i].std()
    print(f"{pn:<10}  uniform: {u_med:.4f} ± {u_std:.4f}   "
          f"informative: {i_med:.4f} ± {i_std:.4f}  (true: {true_values[i]})")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Marginal posterior for mu
ax = axes[0]
ax.hist(flat_unif[:, 1], bins=50, density=True, histtype='step',
        color='steelblue', lw=2, label='Uniform prior on μ')
ax.hist(flat_info[:, 1], bins=50, density=True, histtype='step',
        color='darkorange', lw=2, label='Gaussian prior N(5.1,0.2²)')
t_mu = np.linspace(4.4, 5.7, 300)
ax.plot(t_mu, norm.pdf(t_mu, mu_prev, sigma_prev), 'g--', lw=1.5, label='Prior PDF')
ax.axvline(mu_true, color='red', ls=':', lw=1.5, label=f'True μ = {mu_true}')
ax.set_xlabel(r'$\mu$'); ax.set_ylabel('Density')
ax.set_title(r'Posterior for $\mu$'); ax.legend(fontsize=9)

# Marginal posterior for theta
ax = axes[1]
ax.hist(flat_unif[:, 0], bins=50, density=True, histtype='step',
        color='steelblue', lw=2, label='Uniform prior')
ax.hist(flat_info[:, 0], bins=50, density=True, histtype='step',
        color='darkorange', lw=2, label='Informative prior on μ')
ax.axvline(theta_true, color='red', ls=':', lw=1.5, label=f'True θ = {theta_true}')
ax.set_xlabel(r'$\theta$'); ax.set_ylabel('Density')
ax.set_title(r'Marginal posterior for $\theta$'); ax.legend(fontsize=9)

plt.tight_layout(); plt.show()

**Observations:**
- With $n = 2000$ and $\sigma_{\rm prev} = 0.2$, the data are still dominant; the effect
  would be more pronounced for, e.g., smaller datasets.
- The informative Gaussian prior on $\mu$ shifts and narrows the posterior for $\mu$:
  the posterior is now a compromise between the data likelihood and the prior, with the
  prior pulling toward 5.1.
- The posterior on $\theta$ is slightly narrower when $\mu$ is better constrained,
  because the nuisance uncertainty on $\mu$ contributes less.


---
## Exercise 3 – MCMC diagnostics: integrated autocorrelation time

**Task:** Compute $\tau$ for each parameter and verify the chain is long enough.

In [ ]:
# Run a longer chain to allow reliable autocorrelation estimation
nburn_diag  = 500
nsteps_diag = 4000
nwalkers    = 32
ndim        = 4

map_diag = map_unif.copy()
rng_diag = np.random.default_rng(seed=10)
p0_diag  = map_diag + 1e-3 * rng_diag.standard_normal((nwalkers, ndim))

sampler_diag = emcee.EnsembleSampler(nwalkers, ndim, log_posterior_uniform, args=(x_data,))
state = sampler_diag.run_mcmc(p0_diag, nburn_diag, progress=False)
sampler_diag.reset()
sampler_diag.run_mcmc(state, nsteps_diag, progress=False)

print(f"Acceptance fraction: {sampler_diag.acceptance_fraction.mean():.3f}")

try:
    tau = sampler_diag.get_autocorr_time(quiet=True)
    param_names_diag = ['theta', 'mu', 'sigma', 'lambda']
    print("\nIntegrated autocorrelation times (tau):")
    for pn, t in zip(param_names_diag, tau):
        print(f"  {pn:<8}: tau = {t:.1f},  nsteps/50tau = {nsteps_diag/(50*t):.2f}"
              f"  ({'OK' if nsteps_diag > 50*t else 'NEED MORE STEPS'})")
    print(f"\nRecommended minimum nsteps: {int(50 * max(tau)) + 1}")
except emcee.autocorr.AutocorrError as e:
    print(f"Autocorrelation estimation failed (chain may be too short): {e}")

In [ ]:
# Visualise autocorrelation function for theta
chain = sampler_diag.get_chain()[:, :, 0]   # shape (nsteps, nwalkers)
mean_chain = chain.mean(axis=1)              # mean over walkers

from emcee.autocorr import function_1d
acf = function_1d(mean_chain)

fig, ax = plt.subplots(figsize=(8, 4))
lags = np.arange(len(acf))
ax.plot(lags[:200], acf[:200], 'steelblue', lw=1.5, label=r'ACF of $\theta$')
ax.axhline(0, color='black', lw=0.8)
ax.axhline(1/np.e, color='red', ls='--', lw=1.5, label=r'$1/e$ reference')
ax.set_xlabel('Lag (steps)'); ax.set_ylabel('Autocorrelation')
ax.set_title(r'Autocorrelation function – $\theta$')
ax.legend(); ax.set_xlim(0, 200)
plt.tight_layout(); plt.show()

**Observations:**
- With `nsteps = 4000` and 32 walkers the chain typically satisfies
  $n_{\rm steps} \gg 50\,\tau$ for all parameters.
- The autocorrelation time $\tau$ quantifies how many steps are needed for the chain to
  "forget" its starting point.  If $n_{\rm steps} < 50\,\tau$ the chain is too short
  and the samples are correlated, leading to biased posterior estimates.
- The default `nsteps = 3000` in notebook 03 is already adequate; the check simply
  makes this quantitative.

---
## Exercise 4 – Credible vs. confidence intervals

**Task:** Compare the 95 % Bayesian credible interval for $\theta$ with the 95 % 
frequentist CI from notebook 02.

In [ ]:
from scipy.stats import chi2

# ── Bayesian credible interval (from MCMC) ────────────────────────────────────
theta_samples = flat_unif[:, 0]
ci_bayes_lo, ci_bayes_hi = np.percentile(theta_samples, [2.5, 97.5])

# ── Frequentist profile-likelihood CI ─────────────────────────────────────────
def nll_full(params, data):
    theta, mu, log_sigma, log_lam = params
    sigma = np.exp(log_sigma); lam = np.exp(log_lam)
    if not (0.0 < theta < 1.0): return np.inf
    ls = np.log(theta)       + norm.logpdf(data, mu, sigma)
    lb = np.log(1.0 - theta) + expon.logpdf(data, scale=1.0/lam)
    return -np.sum(np.logaddexp(ls, lb))

x0_mle = [0.5, x_data.mean(), np.log(0.4), np.log(0.4)]
res_mle = minimize(nll_full, x0=x0_mle, args=(x_data,), method='Nelder-Mead',
                   options={'xatol':1e-6,'fatol':1e-6,'maxiter':50000})
mle_internal = res_mle.x; nll_min = res_mle.fun

cl95_1d = chi2.ppf(0.95, df=1) / 2

def profile_nll_theta(theta_val, data, mle_int):
    free_idx = [1, 2, 3]; x0_f = mle_int[free_idx]
    def obj(fv):
        p = mle_int.copy(); p[free_idx] = fv; p[0] = theta_val
        th, m, ls, ll = p
        if not (0.0 < th < 1.0): return 1e10
        s, l = np.exp(ls), np.exp(ll)
        ls2 = np.log(th) + norm.logpdf(data, m, s)
        lb2 = np.log(1-th) + expon.logpdf(data, scale=1.0/l)
        return -np.sum(np.logaddexp(ls2, lb2))
    r = minimize(obj, x0=x0_f, method='Nelder-Mead',
                 options={'xatol':1e-5,'fatol':1e-5,'maxiter':10000})
    return r.fun

grid_theta = np.linspace(0.12, 0.55, 80)
dnll_th = np.array([profile_nll_theta(t, x_data, mle_internal) for t in grid_theta]) - nll_min
in95 = grid_theta[dnll_th <= cl95_1d]
ci_freq_lo, ci_freq_hi = in95.min(), in95.max()

print("=== 95 % intervals for theta ===")
print(f"  Bayesian credible interval: [{ci_bayes_lo:.4f}, {ci_bayes_hi:.4f}]  "
      f"width = {ci_bayes_hi-ci_bayes_lo:.4f}")
print(f"  Frequentist profile-LH CI:  [{ci_freq_lo:.4f}, {ci_freq_hi:.4f}]  "
      f"width = {ci_freq_hi-ci_freq_lo:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
t = np.linspace(0.1, 0.55, 400)
ax.hist(theta_samples, bins=60, density=True, histtype='stepfilled',
        alpha=0.35, color='steelblue', label='Marginal posterior')
ax.axvspan(ci_bayes_lo, ci_bayes_hi, alpha=0.2, color='steelblue',
           label=f'95% Bayesian CI: [{ci_bayes_lo:.3f}, {ci_bayes_hi:.3f}]')
ax.axvspan(ci_freq_lo, ci_freq_hi, alpha=0.15, color='darkorange',
           label=f'95% Frequentist CI: [{ci_freq_lo:.3f}, {ci_freq_hi:.3f}]')
ax.axvline(theta_true, color='red', ls='--', lw=2, label=f'True θ = {theta_true}')
ax.set_xlabel(r'$\theta$'); ax.set_ylabel('Posterior density')
ax.set_title(r'95 % Bayesian credible interval vs. frequentist CI')
ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

**Observations:**
- For large samples and flat priors the Bayesian credible interval and the frequentist
  confidence interval are **numerically very similar** for this model.
- This equivalence holds when the posterior is approximately Gaussian and the prior is
  flat (Bernstein–von Mises theorem).
- They differ in interpretation: the CI is a statement about the procedure (95 % of such
  intervals contain the true value), while the credible interval is a statement about
  the parameter given the data.

---
## Exercise 5 – Marginalising a nuisance parameter on a 2-D grid

**Task:** Fix $\mu$ and $\sigma$ to truth, keep $\theta$ and $\lambda$ free.
Evaluate the posterior $p(\theta,\lambda|\mathbf{x})$ on a 2-D grid and obtain
the marginal $p(\theta|\mathbf{x})$ by numerical integration.

In [ ]:
# 2-D posterior grid (theta, lambda) with mu, sigma fixed to truth
theta_grid  = np.linspace(0.05, 0.60, 80)
lambda_grid = np.linspace(0.25, 0.90, 80)

log_post_grid = np.zeros((len(lambda_grid), len(theta_grid)))

for j, lam_val in enumerate(lambda_grid):
    for i, th_val in enumerate(theta_grid):
        if not (0.0 < th_val < 1.0):
            log_post_grid[j, i] = -np.inf
            continue
        # Log-likelihood with mu, sigma fixed to truth
        ls = np.log(th_val)       + norm.logpdf(x_data, mu_true, sigma_true)
        lb = np.log(1.0 - th_val) + expon.logpdf(x_data, scale=1.0/lam_val)
        log_post_grid[j, i] = np.sum(np.logaddexp(ls, lb))

# Normalise (subtract max for numerical stability, then exponentiate)
log_post_grid -= log_post_grid.max()
post_grid = np.exp(log_post_grid)

# Marginal over lambda: integrate along the lambda axis (axis=0)
dlam = lambda_grid[1] - lambda_grid[0]
marginal_theta = post_grid.sum(axis=0) * dlam

# Normalise the marginal
dtheta = theta_grid[1] - theta_grid[0]
marginal_theta /= (marginal_theta.sum() * dtheta)

# Credible interval from the marginal
cdf_theta = np.cumsum(marginal_theta) * dtheta
lo_idx = np.searchsorted(cdf_theta, 0.025)
hi_idx = np.searchsorted(cdf_theta, 0.975)
ci_grid_lo = theta_grid[lo_idx]
ci_grid_hi = theta_grid[hi_idx]

print(f"Grid-marginal 95% CI for theta: [{ci_grid_lo:.4f}, {ci_grid_hi:.4f}]")
print(f"MCMC credible interval:          [{ci_bayes_lo:.4f}, {ci_bayes_hi:.4f}]")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: 2-D posterior surface
ax = axes[0]
pcm = ax.contourf(theta_grid, lambda_grid, post_grid, levels=30,
                  cmap='Blues', alpha=0.9)
plt.colorbar(pcm, ax=ax, label='Posterior density')
ax.contour(theta_grid, lambda_grid, post_grid,
           levels=[np.exp(-2.0), np.exp(-0.5)], colors=['red', 'orange'])
ax.plot(theta_true, lambda_true, 'r+', ms=14, mew=2, label='True (θ, λ)')
ax.set_xlabel(r'$\theta$'); ax.set_ylabel(r'$\lambda$')
ax.set_title(r'Joint posterior $p(\theta,\lambda|\mathbf{x})$  (μ, σ fixed)')
ax.legend()

# Right: marginal posterior for theta
ax = axes[1]
ax.plot(theta_grid, marginal_theta, 'steelblue', lw=2, label='Grid marginal')
ax.axvspan(ci_grid_lo, ci_grid_hi, alpha=0.2, color='steelblue',
           label=f'95% CI: [{ci_grid_lo:.3f}, {ci_grid_hi:.3f}]')
ax.axvline(theta_true, color='red', ls='--', lw=2, label=f'True θ = {theta_true}')
ax.set_xlabel(r'$\theta$'); ax.set_ylabel('Marginal density')
ax.set_title(r'Marginal posterior $p(\theta|\mathbf{x})$')
ax.legend(fontsize=9)

plt.tight_layout(); plt.show()

**Observations:**
- The 2-D grid approach correctly recovers the marginal posterior for $\theta$ by
  summing (integrating) over the $\lambda$ axis.
- The resulting 95 % credible interval agrees closely with the MCMC result when $\mu$
  and $\sigma$ are fixed.
- MCMC is more practical for higher-dimensional problems where grid methods become
  computationally infeasible (the curse of dimensionality).